# COMP5339 Assignment 2 — Electricity Sector Data Streaming and Analysis
**This notebook covers:**
- **Task 1**: Retrieve power, emissions, and market data from OpenElectricity API
- **Task 2**: Integrate, clean, and cache data to CSV
- **Task 3**: Publish integrated records as MQTT messages (≥0.1 s interval)
- **Task 6**: Continuously loop to simulate an unbounded data stream (60 s rounds)
---
## Configuration
Edit the values below, then **Run All**.

In [ ]:
import os

# OpenElectricity API key (free account: 500 requests/day)
os.environ["OPENELECTRICITY_API_KEY"] = "your_api_key"
API_KEY = os.getenv("OPENELECTRICITY_API_KEY")

# Date range for data retrieval
DATE_START = "2026-05-01"
DATE_END   = "2026-05-08"

# NEM facility codes (10 facilities covering diverse fuel types)
FACILITY_CODES = [
    "APPIN",      # gas_wcmg
    "DEIBDL",     # gas_ocgt
    "BARCALDN",   # gas_ccgt
    "BAYSW",      # coal_black
    "ANGASTON",   # distillate
    "ADP",        # battery / solar
    "APS",        # coal_brown
    "ARWF",       # wind
    "AWABAREF",   # bioenergy_biogas
    "BANKSPT",    # distillate
]

# MQTT broker settings (Tasks 3 & 6)
MQTT_BROKER_HOST = "localhost"
MQTT_BROKER_PORT = 1883
MQTT_TOPIC = "openelectricity/nem/facility_power_emissions"
MQTT_USERNAME = os.getenv("MQTT_USERNAME")
MQTT_PASSWORD = os.getenv("MQTT_PASSWORD")
PUBLISH_DELAY_SECONDS = 0.1         # per-message delay
LOOP_DELAY_SECONDS = 60             # delay between publishing rounds
FACILITY_ID_MODE = "unit_code"      # "unit_code" or "facility_code"
PUBLISH_ONLY_CHANGED_VALUES = False  # skip duplicate values when True

# CSV output paths (Task 2)
COMBINED_CSV = "combined_power_emissions.csv"
MARKET_CSV   = "market_price_demand.csv"

In [ ]:
# ============================================================
# Imports
# ============================================================
import requests
import pandas as pd
import json
import time
import uuid
from pathlib import Path

try:
    import paho.mqtt.client as mqtt
except ImportError:
    mqtt = None  # MQTT publishing unavailable if not installed
    print("Warning: paho-mqtt not installed. Run: pip install paho-mqtt")

---
## Task 1: Data Retrieval

Retrieve per-facility power, CO₂ emissions, and market price/demand from the OpenElectricity REST API.

### 1.1  Facility Metadata

Fetch all NEM facilities with unit‑level details (fuel technology, location, status).

In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}"}

resp = requests.get(
    "https://api.openelectricity.org.au/v4/facilities/",
    headers=headers,
    params={"network_id": "NEM"},
    timeout=30,
)
resp.raise_for_status()
fac_data = resp.json()

if not fac_data.get("success"):
    raise RuntimeError(f"Facilities API failed: {fac_data}")

facilities_df = pd.DataFrame(fac_data["data"])
print(f"Retrieved {len(facilities_df)} NEM facilities")

### 1.2  Power Output (5‑minute interval)

In [ ]:
params = [
    ("interval", "5m"),
    ("metrics", "power"),
    ("date_start", DATE_START),
    ("date_end", DATE_END),
]
for code in FACILITY_CODES:
    params.append(("facility_code", code))

resp = requests.get(
    "https://api.openelectricity.org.au/v4/data/facilities/NEM",
    headers=headers,
    params=params,
    timeout=60,
)
resp.raise_for_status()
power_data = resp.json()

if not power_data.get("success"):
    raise RuntimeError("Power API failed")

results = power_data["data"][0]["results"]

rows = []
for r in results:
    unit_code = r["columns"]["unit_code"]
    for item in r["data"]:          # item = [timestamp, power_value]
        rows.append({"unit_code": unit_code, "timestamp": item[0], "power": item[1]})

power_df = pd.DataFrame(rows)
power_df["timestamp"] = pd.to_datetime(power_df["timestamp"], utc=True)
print(f"Power: {power_df.shape[0]:,} records, {power_df['unit_code'].nunique()} units")

### 1.3  CO₂ Emissions (5‑minute interval)

In [ ]:
params = [
    ("interval", "5m"),
    ("metrics", "emissions"),
    ("date_start", DATE_START),
    ("date_end", DATE_END),
]
for code in FACILITY_CODES:
    params.append(("facility_code", code))

resp = requests.get(
    "https://api.openelectricity.org.au/v4/data/facilities/NEM",
    headers=headers,
    params=params,
    timeout=60,
)
resp.raise_for_status()
emis_data = resp.json()

if not emis_data.get("success"):
    raise RuntimeError("Emissions API failed")

results = emis_data["data"][0]["results"]

rows = []
for r in results:
    unit_code = r["columns"].get("unit_code")
    for item in r["data"]:          # item = [timestamp, emissions_value]
        rows.append({"unit_code": unit_code, "timestamp": item[0], "emissions": item[1]})

emissions_df = pd.DataFrame(rows)
emissions_df["timestamp"] = pd.to_datetime(emissions_df["timestamp"], utc=True)
emissions_df = emissions_df.drop_duplicates().dropna(subset=["unit_code", "timestamp"])
print(f"Emissions: {emissions_df.shape[0]:,} records, {emissions_df['unit_code'].nunique()} units")

### 1.4  Market Price & Demand (5‑minute interval)

In [ ]:
params = [
    ("interval", "5m"),
    ("metrics", "price"),
    ("metrics", "demand"),
    ("date_start", DATE_START),
    ("date_end", DATE_END),
    ("primary_grouping", "network_region"),
]

resp = requests.get(
    "https://api.openelectricity.org.au/v4/market/network/NEM",
    headers=headers,
    params=params,
    timeout=60,
)
resp.raise_for_status()
market_data = resp.json()

if not market_data.get("success"):
    raise RuntimeError("Market API failed")

# Parse long-format data: each row has region + timestamp + (price or demand)
rows = []
for block in market_data["data"]:
    metric = block["metric"]  # "price" or "demand"
    for r in block["results"]:
        region = r["columns"].get("region")
        for item in r["data"]:
            rows.append({"region": region, "timestamp": item[0], metric: item[1]})

market_long_df = pd.DataFrame(rows)
market_long_df["timestamp"] = pd.to_datetime(market_long_df["timestamp"], utc=True)

# Pivot long → wide: price and demand become separate columns
market_df = market_long_df.groupby(["region", "timestamp"], as_index=False).first()
print(f"Market: {market_df.shape[0]:,} records, {market_df['region'].nunique()} regions")

---
## Task 2: Data Integration & Caching

Merge power + emissions into a single dataset, clean and normalise, then export CSV for downstream tasks.

In [ ]:
# 2.1  Merge power and emissions on (unit_code, timestamp)
combined_df = pd.merge(power_df, emissions_df, on=["unit_code", "timestamp"], how="outer")

In [ ]:
# 2.2  Clean and standardise
combined_df["power"]     = pd.to_numeric(combined_df["power"],     errors="coerce")
combined_df["emissions"] = pd.to_numeric(combined_df["emissions"], errors="coerce")
combined_df = combined_df.drop_duplicates().dropna(subset=["unit_code", "timestamp"])
combined_df = combined_df.sort_values(["timestamp", "unit_code"]).reset_index(drop=True)
combined_df["power"]     = combined_df["power"].round(2)
combined_df["emissions"] = combined_df["emissions"].round(2)
print(f"Combined: {combined_df.shape[0]:,} rows × {combined_df.shape[1]} cols")

In [ ]:
# 2.3  Clean market data (NEM only — exclude WEM)
market_df["price"]  = pd.to_numeric(market_df["price"],  errors="coerce")
market_df["demand"] = pd.to_numeric(market_df["demand"], errors="coerce")
market_df = market_df[market_df["region"] != "WEM"]  # Western Australia excluded
market_df = market_df.drop_duplicates().dropna(subset=["region", "timestamp"])
market_df = market_df.sort_values(["timestamp", "region"]).reset_index(drop=True)
market_df["price"]  = market_df["price"].round(2)
market_df["demand"] = market_df["demand"].round(2)
print(f"Market: {market_df.shape[0]:,} rows × {market_df.shape[1]} cols")

In [ ]:
# 2.4  Export CSVs (cached for Tasks 3–5)
combined_df.to_csv(COMBINED_CSV, index=False)
market_df.to_csv(MARKET_CSV, index=False)
print(f"Exported: {COMBINED_CSV}  +  {MARKET_CSV}")

In [ ]:
# 2.5  Export facility metadata for the dashboard (Task 5)
def build_unit_metadata(df_facilities):
    """Extract per-unit metadata (fuel tech, region, coordinates) from facilities_df."""
    rows = []
    for _, row in df_facilities.iterrows():
        for unit in row.get("units", []):
            rows.append({
                "facility_code":  row["code"],
                "facility_name":  row["name"],
                "network_region": row.get("network_region"),
                "unit_code":      unit.get("code"),
                "fueltech_id":    unit.get("fueltech_id"),
                "status_id":      unit.get("status_id"),
                "latitude":       row.get("location", {}).get("lat"),
                "longitude":      row.get("location", {}).get("lng"),
            })
    metadata = pd.DataFrame(rows)
    if not metadata.empty:
        metadata["unit_code"] = metadata["unit_code"].astype(str).str.strip()
        metadata = metadata.dropna(subset=["unit_code"]).drop_duplicates("unit_code")
    return metadata

unit_metadata = build_unit_metadata(facilities_df)
unit_metadata.to_csv("unit_metadata_for_dashboard.csv", index=False)
print(f"Exported unit_metadata_for_dashboard.csv ({len(unit_metadata)} units)")

---
## Task 3: MQTT Publishing

Publish each power + emissions record as an MQTT message in event‑time order, with ≥0.1 s delay.
Each message contains: `timestamp`, `facility_id`, `facility_name`, `network_region`, `fuel_tech`, `power_mw`, `emissions_tco2e`.

In [ ]:
def load_unit_metadata():
    """Load facility/unit metadata. Falls back to unit_code if no CSV available."""
    metadata_path = Path("unit_metadata_for_dashboard.csv")
    if metadata_path.exists():
        metadata = pd.read_csv(metadata_path)
        metadata["unit_code"] = metadata["unit_code"].astype(str).str.strip()
        return metadata
    print("Warning: unit_metadata_for_dashboard.csv not found — using unit_code as facility_id")
    return pd.DataFrame()


def load_market_data(path=MARKET_CSV):
    """Load optional market price/demand data for enrichment."""
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    market = pd.read_csv(path)
    required = {"region", "timestamp", "price", "demand"}
    if not required.issubset(market.columns):
        print(f"Warning: {path} missing expected columns — skipping market merge")
        return pd.DataFrame()
    market["timestamp"] = pd.to_datetime(market["timestamp"], errors="coerce")
    market["price"]  = pd.to_numeric(market["price"],  errors="coerce")
    market["demand"] = pd.to_numeric(market["demand"], errors="coerce")
    market = market.dropna(subset=["region", "timestamp"]).drop_duplicates(subset=["region", "timestamp"])
    return market


def prepare_stream_dataframe():
    """Load, clean, enrich, and sort the combined CSV for MQTT publishing."""
    p = Path(COMBINED_CSV)
    if not p.exists():
        raise FileNotFoundError(f"{COMBINED_CSV} not found. Run Task 1 & 2 first.")

    df = pd.read_csv(p)
    required = {"unit_code", "timestamp", "power", "emissions"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df["unit_code"] = df["unit_code"].astype(str).str.strip()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["power"]     = pd.to_numeric(df["power"],     errors="coerce")
    df["emissions"] = pd.to_numeric(df["emissions"], errors="coerce")
    df = df.dropna(subset=["unit_code", "timestamp"]).drop_duplicates(subset=["unit_code", "timestamp"], keep="last")

    # Merge facility metadata
    metadata = load_unit_metadata()
    if not metadata.empty:
        df = df.merge(metadata, on="unit_code", how="left")
    else:
        for col in ["facility_code", "facility_name", "network_region", "fueltech_id", "status_id"]:
            df[col] = None

    df["facility_id"] = df["unit_code"]

    # Enrich with market price/demand
    market = load_market_data()
    if not market.empty:
        df = df.merge(market, on=["timestamp"], how="left", suffixes=("", "_market"))
        df["region"] = df["network_region"]
    else:
        df["region"] = df["network_region"]
        df["price"]  = None
        df["demand"] = None

    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

In [ ]:
def row_to_mqtt_message(row):
    """Convert a DataFrame row into a canonical MQTT JSON payload."""
    return json.dumps({
        "timestamp":       str(row.get("timestamp", "")),
        "facility_id":     str(row.get("facility_id", row.get("unit_code", ""))),
        "facility_name":   str(row.get("facility_name", "")) if pd.notna(row.get("facility_name")) else "",
        "network_region":  str(row.get("network_region", "")) if pd.notna(row.get("network_region")) else "",
        "fuel_tech":       str(row.get("fueltech_id", "")) if pd.notna(row.get("fueltech_id")) else "",
        "power_mw":        float(row.get("power", 0) or 0),
        "emissions_tco2e": float(row.get("emissions", 0) or 0),
        "price_aud_per_mwh": float(row.get("price", 0) or 0) if pd.notna(row.get("price")) else None,
        "demand_mw":         float(row.get("demand", 0) or 0) if pd.notna(row.get("demand")) else None,
    })


def _value_signature(row):
    """Compute a change‑detection signature for a row (power, emissions, price, demand)."""
    power = row.get("power", 0) or 0
    emissions = row.get("emissions", 0) or 0
    price = row.get("price", 0) or 0
    demand = row.get("demand", 0) or 0
    return (round(float(power), 1), round(float(emissions), 1),
            round(float(price), 1),  round(float(demand), 1))


def create_mqtt_client():
    """Create and connect an MQTT client."""
    if mqtt is None:
        raise ImportError("paho-mqtt is required. Install with: pip install paho-mqtt")

    client_id = f"publisher-{uuid.uuid4().hex[:8]}"
    client = mqtt.Client(client_id=client_id)

    if MQTT_USERNAME:
        client.username_pw_set(MQTT_USERNAME, MQTT_PASSWORD)

    client.connect(MQTT_BROKER_HOST, MQTT_BROKER_PORT, keepalive=60)
    print(f"MQTT connected to {MQTT_BROKER_HOST}:{MQTT_BROKER_PORT}")
    return client


def publish_dataframe_once(df, client=None, dry_run=False):
    """
    Publish every record in df as an MQTT message in timestamp order.

    Args:
        df:      DataFrame sorted by timestamp.
        client:  Connected MQTT client (required unless dry_run=True).
        dry_run: Print messages instead of publishing when True.
    """
    if client is None and not dry_run:
        raise ValueError("client is required when dry_run=False")

    published = 0
    last_signatures = {}  # facility_id → last value signature

    for seq, (_, row) in enumerate(df.iterrows(), start=1):
        fid = str(row.get("facility_id", row.get("unit_code", "")))

        # Skip unchanged values when change-detection is enabled
        if PUBLISH_ONLY_CHANGED_VALUES:
            sig = _value_signature(row)
            if fid in last_signatures and sig == last_signatures[fid]:
                continue
            last_signatures[fid] = sig

        payload = row_to_mqtt_message(row)

        if dry_run:
            if published < 3:  # show first 3 messages only
                print(f"[dry-run] {seq}/{len(df)}  payload={payload[:200]}...")
        else:
            client.publish(MQTT_TOPIC, payload)

        published += 1
        time.sleep(PUBLISH_DELAY_SECONDS)  # ≥0.1 s per message

        if published % 500 == 0:
            print(f"Published {published}/{len(df)} messages ...")

    print(f"Round complete: {published} messages published")
    return published

In [ ]:
# Quick dry-run validation (comment out to skip)
df_stream = prepare_stream_dataframe()
print(f"Stream dataframe: {df_stream.shape[0]:,} records ready for publishing")
publish_dataframe_once(df_stream.head(5), dry_run=True)

---
## Task 6: Continuous Execution

Run the publisher in an infinite loop with 60‑second delay between rounds, simulating an unbounded data stream. Press **Ctrl+C** to stop.

In [ ]:
def run_continuous_publisher():
    """
    Continuously load the latest cached CSV and publish all records.
    Re-loads the CSV each round so the dashboard always gets fresh data.
    """
    round_num = 0

    client = create_mqtt_client()

    print(f"Continuous mode: Topic={MQTT_TOPIC}, Interval={LOOP_DELAY_SECONDS}s")
    print("Press Ctrl+C to stop.")

    try:
        while True:
            round_num += 1
            df = prepare_stream_dataframe()
            print(f"\n=== Round {round_num} — {df.shape[0]:,} records ===")
            publish_dataframe_once(df, client=client)
            print(f"Waiting {LOOP_DELAY_SECONDS}s until next round...")
            time.sleep(LOOP_DELAY_SECONDS)
    except KeyboardInterrupt:
        print(f"\nStopped after {round_num} round(s).")
        client.disconnect()


# Uncomment the line below to start continuous publishing:
# run_continuous_publisher()